# 02. Stage 1A: Content-Based Retrieval (BM25)
 
 Notebook này xây dựng tầng lọc thô dựa trên nội dung (Content-Based) để đề xuất các ứng viên ban đầu cho người dùng sử dụng thuật toán BM25.
 
 ---
 
 ### Phân tích Quyết định Thiết kế:
 *   **Tại sao chọn BM25 thay vì TF-IDF?**
     *   BM25 tích hợp hai cơ chế tiên tiến hơn TF-IDF truyền thống: **Bão hòa Tần suất từ (TF Saturation)** giúp giới hạn tầm ảnh hưởng của một từ khóa lặp lại quá nhiều lần, và **Chuẩn hóa Độ dài Tài liệu (Document Length Normalization)** giúp cân bằng điểm số giữa các phim có metadata ngắn gọn và phim có metadata dài dòng.
 *   **Tại sao không chọn Deep Learning (Sentence-BERT)?**
     *   Sentence-BERT (SBERT) là mạng Transformer dùng để hiểu **ngữ nghĩa tự nhiên** của các câu văn tự do (như `overview`). Với dữ liệu từ khóa rời rạc (như genres hay tên diễn viên), SBERT không mang lại lợi ích về ngữ nghĩa mà còn gây ra Overhead tính toán cực lớn (tải model ~400MB, suy luận chậm trên CPU). BM25 là đủ và hiệu quả hơn rất nhiều cho keyword matching.
 *   **Tại sao không chọn Word2Vec / FastText?**
     *   Các mô hình Word Embedding tĩnh yêu cầu khối lượng văn bản cực lớn để huấn luyện các mối quan hệ từ vựng, hoặc nếu dùng pre-trained thì thường không tối ưu cho các danh từ riêng (tên đạo diễn, diễn viên) hay thuật ngữ điện ảnh đặc thù.
 

### Bước 1: Khởi tạo và Xây dựng Metadata Soup
Tế bào này tải tập dữ liệu phim từ file CSV, điền các giá trị trống (NaN) bằng chuỗi rỗng để tránh lỗi tính toán. Sau đó, ta định nghĩa hàm `build_metadata_soup` để gộp 4 thuộc tính nội dung quan trọng của phim:
*   Thể loại (`genres` - phân tách bằng khoảng trắng thay vì dấu `|`)
*   Đạo diễn (`director` - viết liền không dấu cách để gom cụm chính xác tên)
*   Diễn viên chính (`cast` - lấy tối đa 5 diễn viên đầu tiên)
*   Từ khóa cốt truyện (`keywords`)

Chuỗi gộp này (Metadata Soup) đóng vai trò như một văn bản mô tả ngắn về thuộc tính phim, giúp thuật toán trích xuất từ khóa tính toán độ tương đồng.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
 
# Thêm đường dẫn cha để import recsys_utils
sys.path.append(os.path.abspath('..'))
from recsys_utils import BM25
 
# Load phim
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
 
# Xử lý missing values
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')
movies_df['keywords'] = movies_df['keywords'].fillna('')
 
# 1. Kết hợp đặc trưng dạng văn bản
def build_metadata_soup(row):
    genres = row['genres'].replace('|', ' ')
    cast = ' '.join(row['cast'].split('|')[:5])
    keywords = row['keywords'].replace('|', ' ')
    director = row['director'].replace(' ', '')
    return f"{genres} {director} {cast} {keywords}"
 
movies_df['soup'] = movies_df.apply(build_metadata_soup, axis=1)
display(movies_df[['title', 'soup']].head(3))
 

### Bước 2: Huấn luyện mô hình BM25 và Tính toán TF-IDF Matrix

#### Nguyên lý thuật toán BM25:
BM25 (Best Matching 25) là giải thuật xếp hạng văn bản tiên tiến dựa trên lý thuyết xác xuất. Điểm BM25 của bộ phim (tài liệu $D$) đối với lịch sử người dùng (truy vấn $Q$) được tính bằng:
$$\text{Score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$
Trong đó:
*   $f(q_i, D)$ là tần suất xuất hiện của từ khóa $q_i$ trong soup của phim $D$.
*   $|D|$ và $\text{avgdl}$ lần lượt là độ dài soup phim $D$ và độ dài trung bình của tất cả các phim.
*   $k_1$ là tham số bão hòa tần suất từ (thường chọn $1.2 \le k_1 \le 2.0$). Giới hạn ảnh hưởng của một từ lặp đi lặp lại.
*   $b$ là tham số chuẩn hóa độ dài tài liệu (thường chọn $b = 0.75$). Phạt các phim có soup quá dài dòng chứa nhiều từ rác.
*   $\text{IDF}(q_i)$ là lượng thông tin nghịch đảo của từ khóa:
$$\text{IDF}(q_i) = \ln \left( \frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1 \right)$$

#### So sánh các giải pháp trích xuất đặc trưng nội dung:
| Giải pháp | Nguyên lý | Ưu điểm | Nhược điểm |
| :--- | :--- | :--- | :--- |
| **TF-IDF** | Trọng số dựa trên tần suất từ và tần suất tài liệu nghịch đảo | Đơn giản, dễ cài đặt. | Không có bão hòa tần suất từ (TF) và không phạt độ dài tài liệu. |
| **BM25** (Lựa chọn) | Cải tiến TF-IDF bằng bão hòa TF và phạt độ dài tài liệu | Hiệu năng vượt trội cho dữ liệu dạng từ khóa rời rạc. | Cần điều chỉnh siêu tham số $k_1$ và $b$. |
| **Sentence-BERT** (SBERT) | Dùng Transformer mã hóa văn bản thành dense vector | Hiểu ngữ nghĩa tự nhiên của câu tự do (Overview). | Rất chậm khi chạy trên CPU, tốn bộ nhớ, không tối ưu cho danh từ riêng rời rạc. |

Tế bào này fit mô hình BM25 trên soup phim, đồng thời dùng `TfidfVectorizer` sinh ma trận TF-IDF (sẽ dùng để tính cosine similarity đo tính trùng lặp phục vụ thuật toán MMR ở Stage 3). Các mô hình được lưu lại dưới dạng file Pickle.


In [ ]:
# 2. Xây dựng mô hình BM25 và TF-IDF Matrix (TF-IDF dùng cho MMR)
bm25 = BM25()
bm25.fit(movies_df['soup'])
 
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies_df['soup'])
 
print(f"BM25 fitted on {len(movies_df)} movies.")
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
 
# Lưu trữ ma trận tương đồng và vectorizer
os.makedirs("models", exist_ok=True)
with open("models/bm25_model.pkl", "wb") as f:
    pickle.dump(bm25, f)
    
with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
     
with open("models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)
 

### Bước 3: Định nghĩa Hàm gợi ý Content-Based (BM25)
Tế bào này cài đặt hàm `get_content_based_candidates`:
1. Nhận danh sách các ID phim người dùng đã thích (`liked_movie_ids`).
2. Ghép soup của các phim đã thích này lại để làm một câu truy vấn lớn đại diện cho "gu nội dung" của người dùng.
3. Chạy mô hình BM25 để tính toán điểm tương đồng của câu truy vấn này với tất cả các phim khác trong danh mục.
4. Sắp xếp điểm số giảm dần, loại bỏ các phim người dùng đã xem, và trả về Top N phim làm ứng viên Content-Based cho giai đoạn tiếp theo.


In [ ]:
# 3. Định nghĩa hàm gợi ý Content-Based dùng BM25 cho một danh sách phim đã xem
def get_content_based_candidates(liked_movie_ids, top_n=100):
    # Giới hạn tối đa 20 phim tương tác gần nhất để tránh nhiễu
    liked_movie_ids = list(liked_movie_ids)[-20:]
    liked_idx = movies_df[movies_df['movieId'].isin(liked_movie_ids)].index.tolist()
    if not liked_idx:
        return movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(top_n).tolist()
        
    # Tạo query bằng cách gộp soup của các phim đã xem
    liked_soups = movies_df.iloc[liked_idx]['soup'].tolist()
    query = " ".join(liked_soups)
    
    scores = bm25.transform(query)
    sorted_idx = np.argsort(scores)[::-1]
    
    liked_idx_set = set(liked_idx)
    candidate_indices = [idx for idx in sorted_idx if idx not in liked_idx_set]
    
    recommended_movie_ids = movies_df.iloc[candidate_indices]['movieId'].head(top_n).tolist()
    return recommended_movie_ids

# Test thử nghiệm gợi ý
test_likes = [1339713, 1084244]
candidates = get_content_based_candidates(test_likes, top_n=5)
print("Phim đã xem:", movies_df[movies_df['movieId'].isin(test_likes)]['title'].tolist())
print("Gợi ý Content-based:", movies_df[movies_df['movieId'].isin(candidates)]['title'].tolist())
